In [1]:
import importlib
import torch
import numpy as np
import copy
import os
import pandas as pd
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, Subset, random_split

import NeuralNetwork, funcs, setup, plots

importlib.reload(NeuralNetwork);  importlib.reload(funcs)
importlib.reload(plots);          importlib.reload(setup)

from NeuralNetwork import NeuralNetwork

In [2]:
from setup import (HIDDEN_LAYERS, BATCH_SIZE, MAX_CLUSTERS, PHASE2_MIN_NEURONS,
                   PHASE2_MIN_CONNECTIONS, REGROW_FRAC, N_SPAWN, N_TRAIN_EPOCHS,
                   MAX_PRUNE_ROUNDS, SEARCH_MAX_ROUNDS, SEARCH_SUBSET_SIZE,
                   MAX_ALLOWED_ACC_DROP, N_RETRAIN_EPOCHS, PRUNE_FRAC, PRUNE_CON_FRAC,
                   MIN_WIDTH, HP_SEARCH_GRID, HP_SEARCH_GRID_STAGE2)

device = setup.get_device()
N_TRAIN_EPOCHS = N_TRAIN_EPOCHS if device.type == "cuda" else 8  # CPU gets fewer epochs
train_dataset, val_dataset, test_dataset, fresh_dataset, train_loader, val_loader, test_loader, fresh_loader = setup.get_dataloaders()

device being used: cuda
train size: 172800, val size: 48000, fresh size: 19200, test size: 40000


In [3]:
# Create model and train
model = NeuralNetwork(hidden_sizes=HIDDEN_LAYERS, device=device)
baseline_acc = model.train_model(train_loader=train_loader, epochs=N_TRAIN_EPOCHS)

Training:   0%|          | 0/10 [00:00<?, ?epoch/s]

Epoch 1/10:   0%|          | 0/20 [00:00<?, ?batches/s]

Epoch 2/10:   0%|          | 0/20 [00:00<?, ?batches/s]

Epoch 3/10:   0%|          | 0/20 [00:00<?, ?batches/s]

Epoch 4/10:   0%|          | 0/20 [00:00<?, ?batches/s]

...

Epoch 7/10:   0%|          | 0/20 [00:00<?, ?batches/s]

Epoch 8/10:   0%|          | 0/20 [00:00<?, ?batches/s]

Epoch 9/10:   0%|          | 0/20 [00:00<?, ?batches/s]

Epoch 10/10:   0%|          | 0/20 [00:00<?, ?batches/s]

Validation:   0%|          | 0/3 [00:00<?, ?batches/s]

## Prune Neurons and Retrain

## Hyperparameter Search (ALREADY DONE THE CURRENT PARAMETERS IN SETUP ARE THE ONES DEEMED OPTIMAL BY THE TWO CELLS BELOW)

In [4]:
# # Hyperparameter search — uncomment to run
# # Uses a small subset loader (SEARCH_SUBSET_SIZE samples) for fast retraining during search.
# # After finding the best config, update PRUNE_FRAC / PRUNE_CON_FRAC in setup.py
# # and run the normal pruning cell for the full convergence run.
# # Metric: n_clusters × alignment_score × (val_acc / baseline_acc)


# # Small subset for fast retraining — scored on full val_loader
# _search_idx = torch.randperm(len(train_dataset))[:SEARCH_SUBSET_SIZE].tolist()
# search_loader = DataLoader(Subset(train_dataset, _search_idx), batch_size=BATCH_SIZE, shuffle=True)
# print(f"Search loader: {SEARCH_SUBSET_SIZE} samples, {len(search_loader)} batches/epoch "
#       f"(vs {len(train_loader)} full)")

# original_params = sum(p.numel() for p in model.parameters())
# search_results = []

# for config in HP_SEARCH_GRID:
#     pf, pcf = config['prune_frac'], config['prune_con_frac']
#     print(f"\n{'='*60}")
#     print(f"Searching: prune_frac={pf}, prune_con_frac={pcf}  ({SEARCH_MAX_ROUNDS} rounds, {SEARCH_SUBSET_SIZE} samples)")
#     print(f"{'='*60}")
#     params = (SEARCH_MAX_ROUNDS, pf, pcf, REGROW_FRAC, N_RETRAIN_EPOCHS, MAX_ALLOWED_ACC_DROP)
#     candidate = funcs.pruning(
#         copy.deepcopy(model), search_loader, val_loader, params, baseline_acc,
#         use_max_rounds=True, mode='full', fresh_loader=None,
#         min_width=MIN_WIDTH, max_clusters=MAX_CLUSTERS,
#         phase2_min_neurons=PHASE2_MIN_NEURONS, phase2_min_connections=PHASE2_MIN_CONNECTIONS,
#         n_spawn=N_SPAWN)
#     val_acc         = candidate.accuracy(val_loader)
#     n_params        = sum(p.numel() for p in candidate.parameters())
#     _cmap           = getattr(candidate, 'final_cluster_map', None)
#     n_clusters      = len(_cmap) if _cmap else 1
#     alignment_score = getattr(candidate, 'final_alignment_score', 0.0)
#     composite       = n_clusters * alignment_score * (val_acc / baseline_acc)
#     result = {
#         'prune_frac': pf, 'prune_con_frac': pcf,
#         'val_acc': round(val_acc, 4), 'n_clusters': n_clusters,
#         'alignment_score': round(alignment_score, 4),
#         'composite_score': round(composite, 4),
#         'compression': round(original_params / n_params, 2),
#     }
#     search_results.append(result)
#     print(f"\n--- Result: {result}")

# print(f"\n{'='*60}")
# print("HYPERPARAMETER SEARCH COMPLETE")
# print(f"{'='*60}")
# best = max(search_results, key=lambda r: r['composite_score'])
# print(f"Best config: prune_frac={best['prune_frac']}, prune_con_frac={best['prune_con_frac']}")
# print(f"  val_acc={best['val_acc']}  n_clusters={best['n_clusters']}  "
#       f"alignment={best['alignment_score']}  composite={best['composite_score']}  "
#       f"compression={best['compression']}x")
# pd.DataFrame(search_results).sort_values('composite_score', ascending=False)

In [5]:
# # Stage 2 hyperparameter search — run AFTER stage 1
# # Fix the best prune_frac / prune_con_frac from stage 1, search structural params.
# # Update BEST_PF and BEST_PCF below before uncommenting.


# BEST_PF  = 0.025  # <-- paste best prune_frac from stage 1
# BEST_PCF = 0.45   # <-- paste best prune_con_frac from stage 1

# _search_idx = torch.randperm(len(train_dataset))[:SEARCH_SUBSET_SIZE].tolist()
# search_loader = DataLoader(Subset(train_dataset, _search_idx), batch_size=BATCH_SIZE, shuffle=True)

# original_params = sum(p.numel() for p in model.parameters())
# search_results_s2 = []

# for config in HP_SEARCH_GRID_STAGE2:
#     p2n = config['phase2_min_neurons']
#     mw  = config['min_width']
#     print(f"\n{'='*60}")
#     print(f"Stage 2: phase2_min_neurons={p2n}, min_width={mw}  ({SEARCH_MAX_ROUNDS} rounds)")
#     print(f"{'='*60}")
#     params = (SEARCH_MAX_ROUNDS, BEST_PF, BEST_PCF, REGROW_FRAC, N_RETRAIN_EPOCHS, MAX_ALLOWED_ACC_DROP)
#     candidate = funcs.pruning(
#         copy.deepcopy(model), search_loader, val_loader, params, baseline_acc,
#         use_max_rounds=True, mode='full', fresh_loader=None,
#         min_width=mw, max_clusters=MAX_CLUSTERS,
#         phase2_min_neurons=p2n, phase2_min_connections=PHASE2_MIN_CONNECTIONS,
#         n_spawn=N_SPAWN)
#     val_acc         = candidate.accuracy(val_loader)
#     n_params        = sum(p.numel() for p in candidate.parameters())
#     _cmap           = getattr(candidate, 'final_cluster_map', None)
#     n_clusters      = len(_cmap) if _cmap else 1
#     alignment_score = getattr(candidate, 'final_alignment_score', 0.0)
#     composite       = n_clusters * alignment_score * (val_acc / baseline_acc)
#     result = {
#         'phase2_min_neurons': p2n, 'min_width': mw,
#         'val_acc': round(val_acc, 4), 'n_clusters': n_clusters,
#         'alignment_score': round(alignment_score, 4),
#         'composite_score': round(composite, 4),
#         'compression': round(original_params / n_params, 2),
#     }
#     search_results_s2.append(result)
#     print(f"\n--- Result: {result}")

# print(f"\n{'='*60}")
# print("STAGE 2 SEARCH COMPLETE")
# print(f"{'='*60}")
# best_s2 = max(search_results_s2, key=lambda r: r['composite_score'])
# print(f"Best: phase2_min_neurons={best_s2['phase2_min_neurons']}, min_width={best_s2['min_width']}")
# pd.DataFrame(search_results_s2).sort_values('composite_score', ascending=False)

## Best Hyperparameters Found

Run this cell after the search cells above. It reads the search results and automatically picks the best configuration. You can also override any value manually in the commented-out block.

In [6]:
USE_SEARCH_VALUES = 0  # 1 = use HP search results if available; 0 = always use setup.py defaults

# Always start from setup.py defaults
_pf  = PRUNE_FRAC
_pcf = PRUNE_CON_FRAC
_p2n = PHASE2_MIN_NEURONS
_mw  = MIN_WIDTH
_source = "setup.py defaults"

# if USE_SEARCH_VALUES:
#     if 'search_results' in dir() and search_results:
#         best_s1 = max(search_results, key=lambda r: r['composite_score'])
#         _pf  = best_s1['prune_frac']
#         _pcf = best_s1['prune_con_frac']
#         _source = f"HP search Stage 1 (composite={best_s1['composite_score']}, val_acc={best_s1['val_acc']})"

#     if 'search_results_s2' in dir() and search_results_s2:
#         best_s2 = max(search_results_s2, key=lambda r: r['composite_score'])
#         _p2n = best_s2['phase2_min_neurons']
#         _mw  = best_s2['min_width']
#         _source += f" + Stage 2 (composite={best_s2['composite_score']}, val_acc={best_s2['val_acc']})"

print(f"Source : {_source}")
print(f"\n── Parameters for full training run ──────────────────────")
print(f"  prune_frac          = {_pf}")
print(f"  prune_con_frac      = {_pcf}")
print(f"  phase2_min_neurons  = {_p2n}")
print(f"  min_width           = {_mw}")

# ── Manual override (uncomment any line to override) ───────────────────────
# _pf  = 0.025
# _pcf = 0.45
# _p2n = 100
# _mw  = 20

Source : setup.py defaults

── Parameters for full training run ──────────────────────
  prune_frac          = 0.025
  prune_con_frac      = 0.35
  phase2_min_neurons  = 100
  min_width           = 25


## Run Full Pruning

Uses the parameters from the "Best Hyperparameters Found" cell above. Run that cell first (it auto-fills from the search or falls back to setup.py defaults). Then run this cell to train the final model.

In [7]:
use_max_rounds = device.type != "cuda"

prune_parameters = (MAX_PRUNE_ROUNDS, _pf, _pcf, REGROW_FRAC, N_RETRAIN_EPOCHS, MAX_ALLOWED_ACC_DROP)

final_model = funcs.pruning(
    model, train_loader, val_loader, prune_parameters, baseline_acc,
    use_max_rounds=use_max_rounds, mode='full', fresh_loader=fresh_loader,
    min_width=_mw, max_clusters=MAX_CLUSTERS,
    phase2_min_neurons=_p2n, phase2_min_connections=PHASE2_MIN_CONNECTIONS,
    n_spawn=N_SPAWN)


--- Pruning round 1 ---


  0%|          | 0/1 [00:00<?, ?it/s]

Pruned neurons: layer_0: -1, layer_2: -4, layer_4: -2, layer_6: -4  →  [127 → 124 → 126 → 60]
Retraining after pruning...


Training:   0%|          | 0/8 [00:00<?, ?epoch/s]

Epoch 1/8:   0%|          | 0/22 [00:00<?, ?batches/s]

...

Validation:   0%|          | 0/3 [00:00<?, ?batches/s]

Epoch 14/15:   0%|          | 0/22 [00:00<?, ?batches/s]

Validation:   0%|          | 0/3 [00:00<?, ?batches/s]

Epoch 15/15:   0%|          | 0/22 [00:00<?, ?batches/s]

Validation:   0%|          | 0/3 [00:00<?, ?batches/s]

In [8]:
print(f"Test accuracy after pruning: {final_model.accuracy(val_loader):.2f}")

  0%|          | 0/6 [00:00<?, ?it/s]

Test accuracy after pruning: 0.98


In [9]:
torch.save(final_model, "pruned_model.pth")

In [10]:
# Final model summary
print(f"Val accuracy : {final_model.accuracy(val_loader):.4f}")
linear_layers = [l for l in final_model.layer_stack if hasattr(l, 'out_features')]
sizes = ' → '.join(str(l.out_features) for l in linear_layers)
total_connections = sum((l.weight.data != 0).sum().item() for l in linear_layers)
print(f"Architecture : {sizes}")
print(f"Connections  : {total_connections} non-zero")

if (hasattr(final_model, 'final_cluster_map') and final_model.final_cluster_map
        and len(final_model.final_cluster_map) > 1):
    print(f"\nFinal cluster-digit alignment:")
    funcs.cluster_digit_alignment(
        final_model, val_loader,
        final_model.final_cluster_map, final_model.final_layer_mapping, device)
else:
    print("No multi-cluster structure found in final model.")

  0%|          | 0/6 [00:00<?, ?it/s]

Val accuracy : 0.9807
Architecture : 36 → 26 → 21 → 17 → 10
Connections  : 28496 non-zero

Final cluster-digit alignment:


  0%|          | 0/6 [00:00<?, ?it/s]

  Cluster-digit alignment (mean score=0.511):
              d0    d1    d2    d3    d4    d5    d6    d7    d8    d9 score
  C2           8   202  4485    61   437   164  2060    16    27    15 0.522
  C4          11  3464   340  3427   191    45     0  3702    80   195 0.395
  C5        4817     6    10     5  1556    64  2175    22    21    75 0.520
  C6           2   124     0     1  1373     1     0   612    42  1961 0.495
  C7           0    11     0     0     0   291     0     6     2    17 0.793
  C8           7     5    54    16   474  1242   348   253   554  2114 0.310
  C9          66   890    33  1262    92  2975   247    22  3983   353 0.337
  C10          0    12     0     0   696     1     0   157     3    17 0.715


Pruning creates topology; retraining creates function. The structural components (clusters) emerge from the weight graph during pruning, but digit specialization only emerges during retraining within that fixed structure. A pruned-but-not-retrained network shows alignment ≈ 0 even when structural isolation is perfect. This mirrors biological findings: connectivity is established before functional maps form.